<a href="https://colab.research.google.com/github/paridhietal/TransferLearning/blob/main/transferLearning2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers --quiet

In [ ]:
import chromadb
import json
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload memory_export.json

# now load it into a variable
with open("memory_export.json", "r") as f:
    memory = json.load(f)

print(f"Loaded {len(memory)} tasks from Notebook 1")

In [ ]:
texts = [m["text"] for m in memory]
print(texts)

In [ ]:
# load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# convert texts to embeddings
embeddings = model.encode(texts)
print(embeddings.shape)

In [ ]:
import os

# Define a path for persistence
CHROMA_DB_PATH = "./chroma_db_data"

# Create the directory if it doesn't exist
os.makedirs(CHROMA_DB_PATH, exist_ok=True)

# Initialize a persistent client
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Delete the collection if it already exists to ensure fresh data with correct schema
try:
    client.delete_collection(name="running")
    print("Existing 'running' collection deleted.")
except Exception as e:
    print(f"Could not delete collection (it might not exist yet): {e}")

# Get or create the collection (it will be created if deleted above)
collection = client.get_or_create_collection(name="running")

collection.add(
    ids        = [m["id"] for m in memory],
    embeddings = embeddings.tolist(),
    documents  = texts,
    metadatas  = [
        {
            "score": m["score"],
            "lesson": m["lesson"],
            "task": m["text"],   # Add 'task' from m["text"]
            "output": m["output"] # Add 'output' from m["output"]
        }
        for m in memory
    ]
)

print(f"Stored {len(memory)} tasks in chromaDB")

In [ ]:
#Retrieval function
def retrieve_relevant_memory(query, top_k=3):
  results = collection.query(
      query_texts=[query],
      n_results=top_k
  )

  hits = []
  for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    hits.append({
        "task": meta["task"],
        "output": meta["output"],
        "lesson": meta["lesson"],
        "score": meta["score"],
        "distance": results["distances"][0][i]
    })
  return hits

#test it
hits = retrieve_relevant_memory("RAG")
for h in hits:
  print(f"[score={h['score']}] {h['task']}\n lesson: {h['lesson']}\n")

In [ ]:
#Add new entries to ChromaDB after each run
def update_memory(new_id, task, output, score, lesson):
  collection.add(
      ids=[str(new_id)],
      documents=[f"{task} {lesson}"],
      metadatas=[{
          "task": task,
          "output": output,
          "lesson": lesson,
          "score": str(score)
      }]
  )
  print(f"Memory updated. Total: {collection.count()} entries")

In [ ]:
#Persist so it survives the session
# For PersistentClient, changes are automatically saved. No explicit .persist() call is needed.
# However, it's good practice to ensure the client is properly shut down if needed, but ChromaDB handles flush automatically.
print("ChromaDB data is being persisted to ./chroma_db_data")